# 📈 Laboratorio de Pronóstico: la serie que sigue

**Diplomado en Inteligencia Artificial · UNAC · Clase 04 · Plantilla del aula**

---

### 🎯 Qué vamos a hacer

- Tomar tres años de ventas diarias de un supermercado —lo mismo que usted ya tiene en un Excel— y **proyectar los próximos 90 días**.
- Ver, en una sola gráfica, **por qué sube en diciembre, por qué el sábado vende más y por qué la quincena mueve la caja**.
- Sacar el pronóstico con **piso y techo**: no un número solo, sino un rango que usted puede llevar a un comité.
- Medir **qué tan honesto es ese rango**, porque un pronóstico sin medición es una opinión bien dibujada.

### 🧭 Cómo se usa

- **Usted no tiene que escribir código.** Ni una línea.
- Se corre **celda por celda** con el botón ▶️ que aparece a la izquierda de cada bloque gris, de arriba hacia abajo.
- O todo de una vez con **Entorno de ejecución → Ejecutar todas**.
- Lo **único** que se cambia en este notebook está en el **PANEL DE CONTROL** (sección 4️⃣). El resto se mira, no se toca.
- Si algo se pone rojo, no se asuste: copie el error completo y péguelo en su Claude.


## 1️⃣ ¿Qué hay que instalar?

Nada. Cero. Esta es la mejor noticia del laboratorio.

La herramienta que vamos a usar se llama **Prophet**, la hizo Meta (los de Facebook e Instagram) para pronosticar sus propias series, y **ya viene instalada en Google Colab**. Por eso la primera celda no instala: simplemente la carga y arranca.

> *Nota para el que pregunte:* Prophet está en "modo mantenimiento" desde la versión 1.4.0 — no le van a agregar funciones nuevas, pero sacaron seis versiones en quince meses para mantenerlo compatible. Para un curso eso es una **ventaja**: la herramienta no cambia debajo de nosotros. **Estable no es muerto.**
>
> *Si algún día Colab dejara de traerlo*, se instala con una línea: `pip install prophet` (baja un binario ya compilado, no demora).

**Qué mirar en el resultado:** que aparezca `Listo` y el número de versión de Prophet. Si sale eso, ya tiene todo lo que necesita.


In [ ]:
import warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# Bajarle el volumen a los mensajes tecnicos: aqui solo queremos lo que sirve.
warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

print("Listo. Prophet quedo cargado y no instalamos nada.")
try:
    from importlib.metadata import version
    print("Version de Prophet en este Colab:", version("prophet"))
except Exception:
    print("(No se pudo leer la version, pero la libreria cargo bien.)")


## 2️⃣ ¿De dónde salen los datos si todavía no traigo los míos?

Los inventamos aquí mismo, con una regla clara y una semilla fija, para que **a todos les salga exactamente lo mismo** y nadie tenga que descargar un archivo.

Son **tres años de ventas diarias** de un supermercado de barrio (1.095 días). Y no son números al azar: adentro les metimos los cuatro comportamientos que cualquier gerente de retail reconoce de inmediato.

- una **tendencia** suave al alza (el negocio crece despacio),
- el **fin de semana**: el sábado es el pico y el lunes es flojo,
- la **quincena**: los días 15, 30, 31 y 1 se siente el pago de nómina,
- el **año**: diciembre se dispara y febrero se cae.

Y encima, **ruido**: días buenos y malos sin explicación, como en la vida real. Por eso el modelo nunca va a acertar perfecto — y eso es lo correcto.

**Qué mirar en el resultado:** la tabla de las primeras filas (dos columnas: `ds` la fecha y `y` la venta) y el promedio diario. Prophet exige que las columnas se llamen así, `ds` y `y`; es su único capricho.


In [ ]:
np.random.seed(42)

fechas = pd.date_range(start="2023-09-01", periods=1095, freq="D")  # 3 anos que terminan "hoy"
dia = np.arange(len(fechas))

# 1) Tendencia: el negocio crece despacio (en miles de pesos por dia)
tendencia = 28000 + 3.2 * dia

# 2) Semana: el sabado es el pico, el lunes es flojo (lunes=0 ... domingo=6)
efecto_semana = np.array([-1200, -1400, -900, 200, 2600, 5200, 700])
semana = efecto_semana[fechas.dayofweek.to_numpy()]

# 3) Quincena: los dias de pago de nomina mueven la caja
quincena = np.where(np.isin(fechas.day.to_numpy(), [1, 15, 16, 30, 31]), 2400, 0)

# 4) Ano: diciembre se dispara, febrero se cae
efecto_mes = {12: 6500, 11: 1200, 1: -1500, 2: -2600}
mes = np.array([efecto_mes.get(m, 0) for m in fechas.month])

ruido = np.random.normal(0, 1500, len(fechas))
ventas_dia = np.maximum(tendencia + semana + quincena + mes + ruido, 1000).round(0)

ventas = pd.DataFrame({"fecha": fechas, "ventas_dia": ventas_dia})
datos = ventas.rename(columns={"fecha": "ds", "ventas_dia": "y"})  # los nombres que Prophet exige

print("Dias de historia:", len(datos))
print("Venta promedio del dia (miles de pesos):", int(datos["y"].mean()))
datos.head()


## 3️⃣ ¿Cómo se ve eso que usted ya tiene en su Excel?

Antes de pronosticar nada, se mira. Siempre. Esta gráfica es el paso que la gente se salta y es el que evita las metidas de pata.

La línea clarita es la venta de **cada día** (se ve peluda, así es la realidad). La línea oscura es el **promedio de la semana**, que es como la ve la gerencia.

**Qué mirar en el resultado:** que la línea oscura sube de izquierda a derecha (hay crecimiento) y que aparecen **tres jorobas grandes** — son los tres diciembres. Eso es estacionalidad anual vista a ojo, sin modelo y sin fórmulas.


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(ventas["fecha"], ventas["ventas_dia"],
         color="#9BB0C4", linewidth=0.8, label="Venta de cada día")
plt.plot(ventas["fecha"], ventas["ventas_dia"].rolling(7).mean(),
         color="#0E2036", linewidth=2, label="Promedio de la semana")
plt.title("Ventas diarias del supermercado · tres años de historia")
plt.xlabel("Fecha")
plt.ylabel("Ventas del día (miles de pesos)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 4️⃣ ¿Qué puede cambiar usted? — PANEL DE CONTROL

> # 🔧 AQUÍ Y SOLO AQUÍ SE CAMBIAN COSAS

Cinco perillas. Se mueven con el mouse, se vuelve a correr esta celda y luego las de abajo. No hay que tocar nada más en todo el notebook.

| Perilla | Qué decide | Consejo de gerente |
|---|---|---|
| **HORIZONTE_DIAS** | Cuántos días hacia adelante se pronostica | 90 días es un trimestre: sano. 365 ya es adivinar. Súbalo a **135** si quiere ver diciembre dentro del pronóstico. |
| **INTERVALO_CONFIANZA** | Qué tan ancha es la banda de piso y techo | Prophet trae **0.80 por defecto**, no 0.95. Aquí lo dejamos en 0.90. |
| **USAR_FESTIVOS_COLOMBIA** | Si el modelo aprende que el 20 de julio no es un lunes normal | Enciéndalo. Es meterle conocimiento del país al modelo. |
| **FLEXIBILIDAD_TENDENCIA** | Qué tan rápido se le permite cambiar de rumbo a la tendencia | Alto = se pega a lo reciente. Bajo = terco pero estable. |
| **CORRER_VALIDACION** | Si medimos el error de verdad (sección 🔟) | Enciéndalo al menos una vez. Se demora 1 o 2 minutos. |

**Qué mirar en el resultado:** el resumen impreso con los valores que quedaron activos. Eso confirma que el panel sí se aplicó.


In [ ]:
#@title 🔧 PANEL DE CONTROL — mueva estas perillas y vuelva a correr { display-mode: "form" }

HORIZONTE_DIAS = 90 #@param {type:"slider", min:30, max:365, step:15}
INTERVALO_CONFIANZA = 0.9 #@param [0.8, 0.9, 0.95] {type:"raw"}
USAR_FESTIVOS_COLOMBIA = True #@param {type:"boolean"}
FLEXIBILIDAD_TENDENCIA = 0.05 #@param {type:"slider", min:0.01, max:0.5, step:0.01}
CORRER_VALIDACION = True #@param {type:"boolean"}

print("Panel aplicado:")
print("- Se pronostican", HORIZONTE_DIAS, "dias hacia adelante")
print("- Banda de confianza del", round(INTERVALO_CONFIANZA * 100), "%")
print("- Festivos de Colombia:", "SI" if USAR_FESTIVOS_COLOMBIA else "NO")
print("- Flexibilidad de la tendencia:", FLEXIBILIDAD_TENDENCIA)
print("- Medir el error al final:", "SI" if CORRER_VALIDACION else "NO")


## 5️⃣ ¿Cómo aprende el modelo lo que ya pasó?

Con dos líneas. Literalmente: se crea el modelo y se le entrega la tabla. Él solo se da cuenta de que existe el sábado, de que existe diciembre y de para dónde va la tendencia. **Nosotros no le decimos nada de eso.**

Lo único que sí le regalamos es el calendario de festivos colombianos, porque un lunes festivo en Colombia no se comporta como un lunes normal y eso el modelo no lo puede adivinar solo.

**Qué mirar en el resultado:** el mensaje de que los festivos entraron y el `Modelo entrenado`. Si los festivos fallan, el laboratorio sigue igual: el modelo funciona sin ellos.


In [ ]:
m = Prophet(
    interval_width=INTERVALO_CONFIANZA,
    changepoint_prior_scale=FLEXIBILIDAD_TENDENCIA,
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
)

if USAR_FESTIVOS_COLOMBIA:
    try:
        m.add_country_holidays(country_name="CO")
        print("Festivos de Colombia cargados: el modelo ya sabe que el 20 de julio no es un lunes normal.")
    except Exception as error:
        print("No se pudieron cargar los festivos de Colombia. Seguimos sin ellos y el modelo funciona igual.")
        print("Detalle tecnico (copielo y pegueselo a su Claude):", error)

m.fit(datos)
print("Modelo entrenado con", len(datos), "dias de historia.")


## 6️⃣ ¿Y cuál es el pronóstico?

Se le pide al modelo un calendario que llega hasta el futuro que definimos en el panel, y él lo llena. Devuelve tres números por cada día:

- **yhat** → el pronóstico, el número del medio;
- **yhat_lower** → el piso, el escenario malo;
- **yhat_upper** → el techo, el escenario bueno.

Un gerente no decide con el número del medio. Decide con el piso: **compre y contrate para el piso, prepárese para el techo.**

**Qué mirar en el resultado:** las últimas filas. La fecha final debe estar 90 días (o lo que usted haya puesto) después del último día de la historia.


In [ ]:
futuro = m.make_future_dataframe(periods=HORIZONTE_DIAS, freq="D")
pronostico = m.predict(futuro)

print("Ultimo dia con datos reales:", datos["ds"].max().date())
print("Ultimo dia pronosticado:  ", pronostico["ds"].max().date())
print("Filas totales (historia + futuro):", len(pronostico))

pronostico[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3)


## 7️⃣ ¿Cómo se ve el pronóstico dibujado?

La línea azul es lo que el modelo cree. La zona sombreada es la banda de piso y techo. Los puntos negros son la realidad que ya pasó.

La **línea roja punteada es "hoy"**: a la izquierda el modelo está explicando el pasado, a la derecha está apostando. Todo lo que está a la derecha de esa línea es una hipótesis, no un dato.

**Qué mirar en el resultado:** que la banda se **abre** a medida que se aleja de la línea roja. Eso es el modelo diciéndole la verdad: mañana lo sé bastante bien, en cuatro meses sé mucho menos.


In [ ]:
fig = m.plot(pronostico, figsize=(12, 5))
eje = fig.gca()
eje.axvline(datos["ds"].max(), color="#D54E48", linestyle="--", linewidth=2)
eje.text(datos["ds"].max(), eje.get_ylim()[1] * 0.97, "  hoy",
         color="#D54E48", fontsize=11, va="top")
eje.set_title("Pronóstico de ventas diarias · con piso y techo")
eje.set_xlabel("Fecha")
eje.set_ylabel("Ventas del día (miles de pesos)")
eje.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 8️⃣ ¿Por qué sube en diciembre y por qué el sábado?

**Esta gráfica es la clase entera.** Si usted se lleva una sola imagen de hoy, que sea esta.

El modelo desarma el pronóstico en las piezas que lo componen, y cada pieza es un concepto de negocio:

- **trend (tendencia)** → para dónde va el negocio en el largo plazo, sin el ruido del día a día.
- **holidays (festivos)** → cuánto pesa un festivo colombiano, si los activó en el panel.
- **weekly (semana)** → el mapa del sábado y del lunes. Esto es turnos y personal.
- **yearly (año)** → diciembre, febrero y la temporada. Esto es compras e inventario.

Las etiquetas salen en inglés porque la librería es de Meta; la traducción está arriba.

**Qué mirar en el resultado:** en **weekly**, que el pico esté el sábado. En **yearly**, que diciembre esté arriba y febrero abajo. El modelo descubrió esos dos hechos solo, mirando las fechas — y son exactamente los dos hechos que un gerente de supermercado ya sabía. Cuando el modelo confirma lo que usted sabe, usted puede empezar a creerle lo que no sabía.


In [ ]:
fig2 = m.plot_components(pronostico, figsize=(11, 9))
plt.tight_layout()
plt.show()


## 9️⃣ ¿Me lo puede dar en una tabla que yo pueda pegar en un correo?

Sí. Una gráfica sirve para entender; una tabla sirve para pedir presupuesto. Aquí están los últimos días del pronóstico con nombres en español: **fecha, pronóstico, piso y techo**.

**Qué mirar en el resultado:** compare `piso` y `techo` de una misma fila. Esa distancia es su margen de error real. Si el piso ya le alcanza para cubrir los costos del día, la decisión es fácil; si solo le alcanza el techo, usted está apostando.


In [ ]:
tabla = pronostico[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(10).copy()
tabla["ds"] = tabla["ds"].dt.strftime("%Y-%m-%d")
tabla[["yhat", "yhat_lower", "yhat_upper"]] = tabla[["yhat", "yhat_lower", "yhat_upper"]].round(0).astype(int)
tabla.columns = ["fecha", "pronostico", "piso", "techo"]

print("Ultimos 10 dias del pronostico (miles de pesos):")
display(tabla.reset_index(drop=True))


## 🔟 ¿Y qué tan bueno es este pronóstico?

Cualquier modelo dibuja una línea bonita. La pregunta de gerente es otra: **¿cuánto se equivoca?**

Para responderla se hace algo honesto: se tapa el último trozo de la historia, se pronostica a ciegas y se compara contra lo que de verdad pasó. Eso se repite varias veces, avanzando en el tiempo. Prophet lo trae listo.

Dos números salen de ahí:

- **error_porcentual (MAPE)** → de cada 100 pesos que pronosticó, cuántos se equivocó. Un 5 % significa que si dijo 30.000 se movió unos 1.500 arriba o abajo.
- **cobertura_de_la_banda (coverage)** → **este es el número que le importa a usted.** Si dice 0.90, quiere decir que **9 de cada 10 veces la realidad cayó dentro de la banda: la banda es honesta.** Si dice 0.60, la banda es un adorno y no se puede usar para decidir.

> ⏳ **Esta celda se demora 1 o 2 minutos.** Está entrenando el modelo varias veces. Paciencia: no toque nada, no la vuelva a correr.

**Qué mirar en el resultado:** el MAPE promedio (esperable entre 4 % y 8 % con estos datos) y la cobertura promedio. Y en la tabla, cómo el error **crece** mientras más lejos se pronostica.


In [ ]:
if CORRER_VALIDACION:
    print("Validando... esto se demora 1 o 2 minutos. Paciencia.")
    cortes = cross_validation(m, initial="730 days", period="90 days",
                              horizon="90 days", disable_tqdm=True)
    metricas = performance_metrics(cortes)

    print("Error promedio (MAPE): {:.1f} %".format(metricas["mape"].mean() * 100))
    print("Cobertura promedio de la banda: {:.2f}".format(metricas["coverage"].mean()))
    print("Lectura: de cada 10 dias, en {:.0f} la realidad cayo dentro del piso y el techo.".format(
        metricas["coverage"].mean() * 10))

    resumen = metricas[["horizon", "mape", "coverage"]].copy()
    resumen["horizon"] = resumen["horizon"].dt.days
    resumen["mape"] = (resumen["mape"] * 100).round(1)
    resumen["coverage"] = resumen["coverage"].round(2)
    resumen.columns = ["dias_hacia_adelante", "error_porcentual", "cobertura_de_la_banda"]
    display(resumen.head(12).reset_index(drop=True))
else:
    print("La validacion esta apagada en el PANEL DE CONTROL.")
    print("Encienda CORRER_VALIDACION, vuelva a correr el panel y regrese a esta celda.")


---

## ⚠️ Las 4 advertencias del gerente

Ahora la parte incómoda. La herramienta es fácil; **el juicio es difícil**. Estas cuatro frases son para repetirlas en voz alta cuando alguien llegue con una gráfica de pronóstico a pedirle plata.

**1. Extrapolar de más.** *"El modelo no sabe que usted va a abrir una sede nueva."* Solo conoce lo que ya pasó. Cualquier decisión que usted todavía no ha tomado no está en esa banda.

**2. Cambio de régimen.** *"Una pandemia, una reforma tributaria o un competidor nuevo rompen la regla que generó los datos, y el modelo sigue dibujando la vieja."* No se va a dar cuenta solo, y va a seguir dibujando con la misma confianza. Ese es el peligro.

**3. Series cortas.** *"Con tres años de datos mensuales el modelo vio diciembre solo tres veces."* Nadie aprende una temporada viéndola tres veces. Para estacionalidad anual se necesitan ciclos completos y de sobra: por eso este laboratorio usa datos **diarios**, no mensuales.

**4. Correlación no es causalidad.** *"El modelo le dice que sube, no por qué sube — y usted decide sobre el por qué."* El pronóstico es un insumo de la decisión, nunca la decisión.

---

### 🗣️ Y la cita que hay que conocer

La propia documentación oficial de Prophet, escrita por Meta, dice que sus intervalos suponen que

> *"el futuro verá la misma frecuencia y magnitud de cambios de tendencia que el pasado"*,

califica ese supuesto de

> *"probablemente no es cierto"*

y advierte, textual:

> *"no debería esperar obtener una cobertura precisa en estos intervalos de incertidumbre."*

Léalo otra vez. **El que hizo la herramienta le está diciendo que no le crea del todo.** Eso no es un defecto del software: es honestidad estadística, y es más de lo que ofrece la mayoría de los tableros que le van a mostrar este año.

Dos detalles finos, para que nadie lo sorprenda:
- El intervalo de Prophet viene en **80 % por defecto**, no en 95 %. Si alguien presenta la banda como "el 95 %" sin haberla cambiado, está equivocado.
- La incertidumbre de la **estacionalidad no entra en la banda** salvo que se active un cálculo más pesado (`mcmc_samples`). O sea: la banda que usted vio arriba es **optimista**, la incertidumbre real es un poco mayor.

Y la lección de la competencia mundial de pronósticos (M5): los cincuenta primeros puestos fueron modelos de machine learning y en promedio le ganaron por **14 %** al mejor modelo estadístico. Pero los métodos simples siguen siendo una línea base fuerte y hay que incluirlos **siempre**: suponer que el modelo más grande pronostica mejor *no siempre es cierto*.


---

## 🔭 Y afuera, ¿qué más hay? (panorama 2026)

Lo que acabamos de correr es la herramienta correcta **para aprender y para decidir con criterio**. Estos son los caminos que le van a ofrecer cuando salga de aquí, y qué contestar.

**`statsforecast` de Nixtla — el que usaría su equipo de datos en producción.** Versión 2.1.1 de julio de 2026, el paquete más activo del grupo, licencia abierta. Trae varios modelos automáticos y uno que a usted le va a servir: **Croston**, hecho para inventarios de baja rotación, esos productos que venden cero muchos días. El propio proyecto afirma ser mucho más rápido que Prophet a la hora de correr miles de series a la vez (es un dato del fabricante, no un estudio independiente). Si su empresa necesita pronosticar 5.000 referencias cada noche, este es el nombre que hay que decir.

**Chronos-2 de Amazon — pronostica sin entrenar.** Es un modelo fundacional de series de tiempo (120 millones de parámetros, licencia abierta): usted le pasa la serie y devuelve el pronóstico **sin ajustar nada**, como quien le pide un resumen a un chat. Es el estado del arte público en 2026 y ya está en producción dentro de AWS. Necesita GPU y trabajo técnico, así que no es un laboratorio de dos horas — pero es la dirección en la que va todo esto.

**Pronóstico en Google Sheets, sin una línea de código.** Desde febrero de 2026, Google Sheets pronostica con TimesFM desde **Connected Sheets**: se abre la hoja conectada, se entra a *Advanced Analytics* y se elige *Create a Forecast*, con horizonte e intervalo de confianza configurables. Es, literalmente, el mensaje de este diplomado: **un gerente pronosticando sin programar.** Advertencia honesta: eso corre sobre BigQuery y en la práctica pide un proyecto de Google Cloud con permisos, así que confirme con su área de TI antes de prometerlo en un comité.

**Y uno que descartamos a propósito:** TimeGPT. El modelo es propietario y de pago, con prueba de 30 días. No sirve para un salón de 16 personas ni para material que se reutiliza cada semestre.


---

## 🏁 Ahora con SUS datos

Todo lo que corrió arriba funciona igual con las ventas de su empresa. Solo hay que cambiar **de dónde salen los datos**, y nada más: el resto del notebook no se toca.

**Lo que necesita:** un Excel o un CSV con **dos columnas y nada más**.

| columna | qué va | ejemplo |
|---|---|---|
| `ds` | la fecha | `2026-01-15` |
| `y` | el número que quiere pronosticar | `31500` |

Puede ser ventas del día, unidades vendidas, pacientes atendidos, tiquetes de soporte, metros cúbicos facturados o consultas al call center. Prophet no sabe qué es: ve una fecha y un número.

**Los tres pasos, en orden:**

1. Guarde su archivo como **CSV** con esas dos columnas exactas (en Excel: *Archivo → Guardar como → CSV UTF-8*).
2. En este notebook, use el ícono de **📁 carpeta** en la barra de la izquierda y arrastre el archivo ahí. O si prefiere el camino con código, cree una celda nueva y pegue esto —son dos líneas, y Colab le abre el botón de "Elegir archivos":

   ```python
   from google.colab import files
   subido = files.upload()
   ```

3. Reemplace la celda 2️⃣ (la que inventa los datos) por la lectura de su archivo:

   ```python
   datos = pd.read_csv("ventas_de_mi_empresa.csv")
   datos["ds"] = pd.to_datetime(datos["ds"])
   datos = datos[["ds", "y"]].sort_values("ds").reset_index(drop=True)
   ```

Y vuelva a correr de la 4️⃣ hacia abajo. Eso es todo.

**Tres advertencias prácticas antes de que lo intente:**
- **Mínimo dos años de historia** si quiere que aprenda la temporada del año. Con menos, el pronóstico corre pero la estacionalidad es frágil.
- **Los días faltantes no son ceros.** Si el negocio cierra los domingos, simplemente no ponga esas filas; no escriba `0`, porque el modelo lo va a creer.
- **Si su serie tiene muchos ceros** (productos de baja rotación), el error porcentual MAPE se rompe y da números absurdos. Ese es el caso de Croston en `statsforecast`, no de Prophet.


### 🤖 Pregúntele a su Claude

No tiene que aprenderse nada de lo que hay aquí. Tiene que aprender a **pedirlo bien**. Abra su Claude, tenga este notebook a mano y use uno de estos encargos como punto de partida:

> **Para adaptar la plantilla:**
> *"Tengo el notebook de pronóstico de la Clase 04 del diplomado, el que usa Prophet en Colab. Quiero adaptarlo a mis datos. Mis columnas son: [descríbalas: nombre exacto, qué contiene, desde qué fecha, con qué frecuencia — diaria, semanal, mensual]. Dime exactamente qué celdas cambio y déjame el código listo para pegar. No cambies el resto del notebook."*

> **Para entender un resultado que no le cuadra:**
> *"Corrí el pronóstico y me dio un MAPE de [X] % y una cobertura de [Y]. Explícame en lenguaje de negocio si eso es bueno o malo para [su industria], y qué debería revisar en mis datos antes de llevar esto a un comité."*

> **Para blindarse antes de presentarlo:**
> *"Voy a presentar este pronóstico a la gerencia de [su empresa]. Hazme las cinco preguntas incómodas que me van a hacer y ayúdame a responderlas, incluyendo qué NO puedo afirmar con este modelo."*

Ese último es el que más le va a servir. **El pronóstico no lo defiende el modelo: lo defiende usted.**


---

**Diplomado en IA · UNAC · Clase 04 · Si algo falla, copie el error completo y pégueselo a su Claude: él sabe de este notebook.**
